# Day 2 — Arrow columnar memory and why Polars is fast

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** [Day 2 — Arrow Columnar Memory: Why Polars Is Fast](https://python.plainenglish.io/day-2-arrow-columnar-memory-why-polars-is-fast-cf7a6c5be111)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Why does the same table take less memory and run faster in Polars? Because of **how a column is laid out in memory**.
We compare three layouts of the same text column:

1. pandas `object` — one Python object per value (the pandas 1.x/2.x default)
2. pandas 3 `str` — Arrow-backed strings (the pandas 3.0 default with `pyarrow` installed)
3. Polars `String` — Arrow memory, managed in Rust

In [2]:
import sys

import pandas as pd
import pyarrow as pa

print("pandas", pd.__version__, "| pyarrow", pa.__version__)

customers = pl.read_csv("data/raw/olist_customers_dataset.csv")
customers.head()

pandas 3.0.6 | pyarrow 26.0.0


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,i64,str,str
"""06b8999e2fba1a1fbc88172c00ba8b…","""861eff4711a542e4b93843c6dd7feb…",14409,"""franca""","""SP"""
"""18955e83d337fd6b2def6b18a428ac…","""290c77bc529b7ac935b93aa66c333d…",9790,"""sao bernardo do campo""","""SP"""
"""4e7b3e00288586ebd08712fdd0374a…","""060e732b5b29e8181a18229c7b0b2b…",1151,"""sao paulo""","""SP"""
"""b2b6027bc5c5109e529d4dc6358b12…","""259dac757896d24d7702b9acbbff3f…",8775,"""mogi das cruzes""","""SP"""
"""4f2d8ab171c80ec8364f7c12e35b23…","""345ecd01c38d18a9036ed96c73b8d0…",13056,"""campinas""","""SP"""


## 1. One column, three layouts

In [ ]:
cities = customers["customer_city"]  # ~99k short strings like "sao paulo"

as_object = pd.Series(cities.to_list(), dtype=object)
as_pd_str = pd.Series(cities.to_list())  # pandas 3 default string dtype

print("pandas object dtype:", as_object.dtype)
print("pandas default dtype:", as_pd_str.dtype)
print("polars dtype:", cities.dtype)

pandas object dtype: object
pandas default dtype: str
polars dtype: String


In [ ]:
layouts = pl.DataFrame(
    {
        "layout": ["pandas object", "pandas 3 str", "polars String"],
        "mb": [
            as_object.memory_usage(deep=True) / 1e6,
            as_pd_str.memory_usage(deep=True) / 1e6,
            cities.estimated_size() / 1e6,
        ],
    }
)
layouts

layout,mb
str,f64
"""pandas object""",5.901405
"""pandas 3 str""",1.824324
"""polars String""",1.028664


## 2. Why `object` is expensive

An `object` column is an array of **pointers**. Each pointer leads to a full Python `str` object somewhere else on the heap, with its own header (type, reference count, length, hash) before the characters even start.

In [ ]:
value = "sao paulo"
print(repr(value), "->", len(value), "characters")
print("Python str object size:", sys.getsizeof(value), "bytes")
print("Pointer in the object array: 8 bytes")

'sao paulo' -> 9 characters
Python str object size: 50 bytes
Pointer in the object array: 8 bytes


## 2b. Pointers vs buffers, in plain words

This is the one idea from today worth really understanding, so here it is without jargon.

A **pointer** is just a memory address: "the thing you want is over there." A **buffer** is one continuous block of memory with the data itself in it.

Think of a warehouse. The pandas `object` layout is a clipboard of 99,000 slips, each saying "box #4,812 is in aisle 7." To read the column you walk to every box, open it, read the label on top (type, size, reference count) and only then the city name. The Arrow layout is one long roll of paper with every city name written end to end, plus a short ruler that says where each name starts. You read it top to bottom in one walk.

```
pandas object column
  [ptr][ptr][ptr][ptr] ...          8 bytes each, one per row
    |    |    |    |
    v    v    v    v                each leads to its own Python object,
  {header|"sao paulo"}              scattered somewhere on the heap,
  {header|"campinas"}               with ~40-50 bytes of header
  ...                               before the first character

Arrow string column (the classic layout)
  validity: 1 1 0 1                 1 bit per row: value or null
  offsets:  0 9 17 17 23            where each value starts and ends
  data:     sao paulocampinassantos ...   all characters, packed together
```

Two things follow from that picture:

- **Less memory:** no header per value, and a null costs one bit instead of an object.
- **Faster reads:** the CPU streams through one block instead of chasing 99,000 addresses.

One thing that does **not** follow, and that I got wrong at first: Arrow does **not** store repeated values once. If `"sao paulo"` appears 15,000 times, all 15,000 copies are in the data buffer. Storing each unique value once is a different feature, `Categorical`/`Enum`, which gets its own post later in the series. The saving here comes from dropping the per-value overhead, not from removing repeats.

Arrow stores the same column as a few **contiguous buffers**: a validity bitmap for nulls, plus the string data and the lengths/offsets that slice it.
Polars uses Arrow's *string view* layout, where short strings (12 bytes or less) sit inline next to their length — no pointer chasing at all.

See `assets/day_002/string_memory.svg` for the picture.

In [ ]:
# Peek at the Arrow memory behind a Polars column
arrow_cities = cities.to_arrow()
print("Arrow type:", arrow_cities.type)
for i, buf in enumerate(arrow_cities.buffers()):
    print(f"buffer {i}:", None if buf is None else f"{buf.size:,} bytes")

Arrow type: large_string
buffer 0: None
buffer 1: 795,536 bytes
buffer 2: 1,028,664 bytes


## 3. Columnar means operations touch only what they need

Computing on one column reads one contiguous block of memory. The CPU can prefetch it and use SIMD instructions.

In [ ]:
import time

items_pl = pl.read_csv("data/raw/olist_order_items_dataset.csv")
items_pd = items_pl.to_pandas()

t0 = time.perf_counter()
for _ in range(50):
    items_pd["price"].sum()
t_pd = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(50):
    items_pl["price"].sum()
t_pl = time.perf_counter() - t0

print(f"50 sums  pandas: {t_pd * 1000:.1f} ms | polars: {t_pl * 1000:.1f} ms")

50 sums  pandas: 6.6 ms | polars: 0.7 ms


On a numeric column both libraries already use one contiguous array (NumPy or Arrow), so **layout is not the reason** for any gap here. Summing ~112k floats takes microseconds; most of a pandas call is overhead around it, including the `skipna=True` default that checks every value for `NaN`. Polars goes straight to Rust, and its validity bitmap already says "no nulls", so it just adds. Expect the gap to shrink on bigger columns.

## 4. Chunks: Arrow columns can be split

A Polars column can be made of several Arrow chunks. `read_csv` parses in parallel, so a freshly read frame is often already several chunks. `vstack` appends chunk lists without copying; `rechunk()` copies everything into one contiguous block.

In [ ]:
print("chunks after read_csv:", customers.n_chunks())  # parallel parsing -> several pieces

stacked = customers.vstack(customers).vstack(customers)  # vstack appends chunk lists, no copy
print("chunks after vstack:", stacked.n_chunks())
print("chunks after rechunk:", stacked.rechunk().n_chunks())

chunks after read_csv: 19
chunks after vstack: 57
chunks after rechunk: 1


## 5. Arrow is a shared language

Because Polars, pandas 3, DuckDB and PyArrow all speak Arrow, data can move between them with little or no copying.

In [ ]:
import duckdb

arrow_table = customers.to_arrow()
back = pl.from_arrow(arrow_table)
print("round trip equal:", back.equals(customers))

# DuckDB can query a Polars DataFrame directly
duckdb.sql("SELECT customer_state, count(*) AS n FROM customers GROUP BY 1 ORDER BY n DESC LIMIT 5").pl()

round trip equal: True


customer_state,n
str,i64
"""SP""",41746
"""RJ""",12852
"""MG""",11635
"""RS""",5466
"""PR""",5045


## How deep should you go? (a note to you, and to me)

Arrow internals are a rabbit hole: offsets, string views, buffer alignment, SIMD, how chunks are scheduled across threads. It is fascinating, and almost none of it changes the Polars code you will write this month.

You need:

- **Pointers vs buffers** — the picture above. Be able to explain it in three sentences.
- **Validity bitmap** — one sentence: nulls are a bitmap, so they are cheap and the same for every type.
- **Chunks** — one sentence: a column can be several pieces, and `rechunk()` glues them into one.

You can safely skip, for now: byte layouts, alignment and padding, how string views are encoded, and when exactly to call `rechunk()`. Performance tuning comes much later in this series, and it will make more sense once you have written real queries.

If you are curious anyway, read the source rather than second-hand summaries (including this one):

- [Arrow Columnar Format specification](https://arrow.apache.org/docs/format/Columnar.html) — the official description of buffers, validity bitmaps and the string view layout.
- [Why we have rewritten the string data type](https://pola.rs/posts/polars-string-type/) — the Polars team on moving to string views.
- [Using StringView / German Style Strings to make Queries Faster](https://datafusion.apache.org/blog/2024/09/13/string-view-german-style-strings-part-2) — a visual walk-through of the same layout from the DataFusion team.

## Glossary

| Term | In one line |
|---|---|
| **Apache Arrow** | A standard way to lay out columnar data in memory, shared by Polars, pandas 3, DuckDB and PyArrow. |
| **Pointer** | A memory address that says where a value lives, instead of the value itself. |
| **Buffer** | One continuous block of memory holding data. An Arrow column is a few of these. |
| **Validity bitmap** | One bit per row marking value (1) or null (0). |
| **Offsets** | In the classic string layout, the positions where each value starts and ends in the data buffer. |
| **String view** | The string layout Polars uses: short strings stored inline next to their length, longer ones as a prefix plus a reference. |
| **Chunk** | One piece of a column. A column can be several chunks glued together; `rechunk()` makes it one. |

## What to take from this post

- A pandas `object` column is an array of **pointers** to separate Python objects; an Arrow column is a few **contiguous buffers**. That is most of the memory story.
- Arrow saves memory by dropping per-value overhead, **not** by storing repeated values once. That is what `Categorical` is for.
- Contiguous memory plus no Python objects is what lets Polars use SIMD and every core. On plain numbers both libraries are contiguous, so any gap there comes from overhead and null handling, not layout.
- Arrow is a shared language: Polars, pandas 3, DuckDB and PyArrow hand data to each other cheaply.

What you do **not** need to remember: byte layouts, offsets vs views, or when to `rechunk()`. Knowing the picture is enough to move on.

## Results for the post

| Layout | Memory for `customer_city` (MB) |
|---|---|
| pandas `object` | 5.90 |
| pandas 3 `str` | 1.82 |
| Polars `String` | 1.03 (`estimated_size()` counts only the characters) |

## Try it yourself

1. Repeat the memory comparison for `review_comment_message` in `olist_order_reviews_dataset.csv` (long text, many nulls). Does the ranking change?
2. Check `customers["customer_zip_code_prefix"].dtype`. What happened to the leading zeros of zip codes like `01409`?
3. Is the 12x float-sum gap about layout? Time `items_pd["price"].sum(skipna=False)` and `items_pd["price"].to_numpy().sum()` and compare with Polars.
